In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.memory import ConversationBufferMemory

load_dotenv()
gemini_api_key=os.getenv('GOOGLE_API_KEY')
tavily_api_key=os.getenv('TAVILY_API_KEY')

llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

qa_prompt=PromptTemplate.from_template('Answer the users queries clearly without hallucination: {query}')
qa_chain=LLMChain(llm=llm,prompt=qa_prompt)
qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.run,
    description='A basic LLM that answers users queries clearly ')

search_tool=Tool(
    name='Tavily Search',
    func=TavilySearchResults(max_results=2).run,
    description='An LLM that searches the web for latest information')

tools=[qa_tool,search_tool]
memory=ConversationBufferMemory(memory_key='chat_history',return_messages=True)

In [2]:
#Agent type 1:
agent=initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False
)

response=agent.invoke({'input':'Summarize LangChain in 2 lines, then tell me who created it'})
print('\nAgent response: ',response)

C:\Users\DELL\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(



Agent response:  {'input': 'Summarize LangChain in 2 lines, then tell me who created it', 'output': 'LangChain is an open-source framework that simplifies building applications with large language models (LLMs) by chaining together various components. It enables developers to connect LLMs with external data and computation, creating more powerful and context-aware AI applications. LangChain was created by Harrison Chase.'}


In [3]:
#Agent type 1:
agent=initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

response=agent.invoke({'input':'Summarize LangChain in 2 lines, then tell me who created it'})
print('\nAgent response: ',response)



> Entering new AgentExecutor chain...
Action: Simple QA
Action Input: Summarize LangChain in 2 lines and tell me who created it.
Observation: LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs) by providing tools to chain together various components. It allows developers to connect LLMs with external data sources and computation, enabling more powerful and context-aware AI applications.

LangChain was created by **Harrison Chase**.
Thought:I now know the final answer
Final Answer: LangChain is an open-source framework that simplifies the development of applications using large language models (LLMs) by chaining together various components. It enables developers to connect LLMs with external data and computation, creating more powerful and context-aware AI applications. LangChain was created by Harrison Chase.

> Finished chain.

Agent response:  {'input': 'Summarize LangChain in 2 lines, then tell me who created i

In [4]:
#Agent type 2:
agent=initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)

response=agent.invoke({'input':'Summarize LangChain in 2 lines, then tell me who created it'})
#print('\nAgent response: ',response)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Summarize LangChain in 2 lines and tell me who created it.
Observation: LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs) by providing tools to chain together various components. It allows developers to connect LLMs with external data sources and computation, enabling more powerful and context-aware AI applications.

LangChain was created by **Harrison Chase**.
Thought:Do I need to use a tool? No
AI: LangChain is an open-source framework that simplifies building applications with large language models by chaining together various components. It helps connect LLMs with external data and computation for more powerful AI applications.

LangChain was created by Harrison Chase.

> Finished chain.


In [5]:
#Agent type 3:
from langchain.tools import StructuredTool
from pydantic.v1 import BaseModel

class TitleInput(BaseModel):
    topic: str
    tone: str='concise'

def title_tool_fn(topic: str, tone: str='concise') -> str:
    return f"{tone.title()} Title: {topic} in Practice"
    
title_tool=StructuredTool.from_function(
    name='TitleMaker',
    func=title_tool_fn,
    description='Generate a title given a topic and a tone',
    args_schema=TitleInput
)

agent=initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

response=agent.invoke({'input':'Make a friendly title about LangGraph tutorials'})
#print('\nAgent response: ',response)



> Entering new AgentExecutor chain...
Thought:The user wants a friendly title about LangGraph tutorials. I should use the `TitleMaker` tool for this. The `topic` will be "LangGraph tutorials" and the `tone` will be "friendly".
Action:
```
{
  "action": "TitleMaker",
  "action_input": {
    "topic": "LangGraph tutorials",
    "tone": "friendly"
  }
}
```
Observation: Friendly Title: LangGraph tutorials in Practice
Thought:The user asked for a friendly title about LangGraph tutorials. I previously used the `TitleMaker` tool with the topic "LangGraph tutorials" and tone "friendly", and the observation was "Friendly Title: LangGraph tutorials in Practice". This is the final answer.
Action:
```
{
  "action": "Final Answer",
  "action_input": "LangGraph tutorials in Practice"
}
```

> Finished chain.


In [7]:
#Agent type 4:
intermediate_tool=Tool(
    name='Intermediate Answer',
    func=TavilySearchResults(max_results=1).invoke,
    description='Use this tool to search for information'
)

agent=initialize_agent(
    tools=[intermediate_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)

response=agent.invoke({'input':'Summarize LangChain in 2 lines, then tell me who created it'})
#print('\nAgent response: ',response)



> Entering new AgentExecutor chain...
Could not parse output: No.
Intermediate answer: Invalid or incomplete response
Could not parse output: Are follow up questions needed here: No.
Intermediate answer: Invalid or incomplete response
Could not parse output: Are follow up questions needed here: No.
Intermediate answer: Invalid or incomplete response
Could not parse output: Are follow up questions needed here: No.
Intermediate answer: Invalid or incomplete response
Could not parse output: Are follow up questions needed here: No.
Intermediate answer: Invalid or incomplete response
Could not parse output: Are follow up questions needed here: No.
Intermediate answer: Invalid or incomplete response
Are follow up questions needed here: No.
So the final answer is: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs). It provides tools and components for chaining together different LLM calls and other computational steps. It was create